**Behavioral Tool**

γshort = 1.0694 + 0.0045 × debtrules3 − 0.0067 × debtnorms8,

αshort = 0.501796 + 0.06459 × debtrules16 − 0.034671 × debtpers5 − 0.044603 × debtnorms9, 

λshort = 1.177504 − 0.025919 × debtrules16 + 0.028538 × debtpers9,

δshort = 0.050555 − 0.003716 × debtpers9 + 0.001902 × debtpers11 − 0.000938 × debtrules9. 

**Financial Tool:**
Credit Scoring ML Model

**Regulations and Policy Retreiver Tool:**
Hybrid Search

**Macro Data Tool:**
Web Search Tool

**Calculator Tool**

**Clustering**
- Behavioral Scoring
- Adaptive Questioning
- Assign Cluster (Confidence)

In [14]:
from tools.behavioral_scoring import behavioral_scoring

behavioral_scoring(2, 3, 4, 1, 3, 5, 1, 3)

(1.0582999999999998, 0.5916760000000001, 1.216518, 0.031063000000000004)

In [15]:
import os

print("Current directory:")
print(os.getcwd())

print("\nFiles:")
print(os.listdir())

Current directory:
e:\LangGraph-RAG-Agent-main\LangGraph-RAG-Agent-main\backend\app

Files:
['auth', 'base.ipynb', 'chat', 'config.py', 'db', 'documents', 'lifespan.py', 'main.py', 'middleware.py', 'threads', 'tools', 'users', '__init__.py', '__pycache__']


In [16]:
import os

print(os.listdir("tools"))



['behavioral_scoring.py', 'financial_scoring.py', 'macro_economy_api.py', '__init__.py', '__pycache__']


In [22]:
import joblib

kmeans_loaded = joblib.load("./final_kmeans_model.joblib")

kmeans_loaded.predict([[1.0582999999999998, 0.5916760000000001, 0.031063000000000004]])


FileNotFoundError: [Errno 2] No such file or directory: './final_kmeans_model.joblib'

### Macro API Fetch

In [5]:
from tools.macro_economy_api import MacroEconomicFetcher

result = MacroEconomicFetcher().get_macro_context("GRC")    

print(result)

REQUEST URL: https://api.worldbank.org/v2/country/GRC/indicator/FP.CPI.TOTL.ZG?format=json&per_page=100
REQUEST URL: https://api.worldbank.org/v2/country/GRC/indicator/SL.UEM.TOTL.ZS?format=json&per_page=100
REQUEST URL: https://api.worldbank.org/v2/country/GRC/indicator/FR.INR.RINR?format=json&per_page=100
REQUEST URL: https://api.worldbank.org/v2/country/GRC/indicator/NY.GDP.MKTP.KD.ZG?format=json&per_page=100
{'inflation_rate': 2.48, 'unemployment_rate': 8.54, 'real_interest_rate': 0.0, 'gdp_growth': 2.07}


### Financial Scoring


In [6]:
from tools.financial_scoring import calculate_financial_wellbeing
result = calculate_financial_wellbeing(
        dti=0.34,                     # from statement or conversation
        cash_buffer_months=1.6,       # from statement or user estimate
        savings_rate=0.07,            # from statement or user estimate
        late_frequency=0.10,          # from statement or user estimate
        high_interest_debt_share=0.45,# from statement or user estimate
        source="mixed"
    )
result

{'financial_wellbeing_score': 52,
 'category': 'Stressed',
 'breakdown': {'dti_score': 12,
  'cash_buffer_score': 10,
  'savings_rate_score': 10,
  'late_frequency_score': 12,
  'high_interest_debt_score': 8},
 'inputs': {'dti': 0.34,
  'cash_buffer_months': 1.6,
  'savings_rate': 0.07,
  'late_frequency': 0.1,
  'high_interest_debt_share': 0.45,
  'source': 'mixed'}}

In [7]:
import os

print(os.listdir())

['auth', 'base.ipynb', 'chat', 'config.py', 'db', 'documents', 'lifespan.py', 'main.py', 'middleware.py', 'threads', 'tools', 'users', '__init__.py', '__pycache__']


In [8]:
from typing import Annotated, Sequence, TypedDict
from langgraph.graph import StateGraph, START, END
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage, BaseMessage  # Message for providing instructions to the LLM
from langgraph.graph.message import add_messages
from pydantic import BaseModel, Field
from langchain_core.tools import StructuredTool, tool


class AgentState(TypedDict):
    messages: Annotated[Sequence[BaseMessage], add_messages]

class BehavioralScoringInput(BaseModel):
    debtrules3: int = Field(description="Likert response from 1 to 5")
    debtnorms8: int = Field(description="Likert response from 1 to 5")
    debtrules16: int = Field(description="Likert response from 1 to 5")
    debtpers5: int = Field(description="Likert response from 1 to 5")
    debtnorms9: int = Field(description="Likert response from 1 to 5")
    debtpers9: int = Field(description="Likert response from 1 to 5")
    debtpers11: int = Field(description="Likert response from 1 to 5")
    debtrules9: int = Field(description="Likert response from 1 to 5")

behavioral_scoring_tool = StructuredTool.from_function(
    func=behavioral_scoring,
    name="behavioral_scoring",
    description="Calculate the user's behavioral preferences from 8 Likert-scale responses.",
    args_schema=BehavioralScoringInput,
)

@tool
def macro_economic_context_tool(country_code: str):
    """
    Fetch the latest available macroeconomic indicators for a country.

    Args:
        country_code: ISO 2-letter country code, e.g. 'GRC' for Greece,
                      'US' for United States, or 'gb' for United Kingdom.

    Returns:
        Dictionary containing inflation rate, unemployment rate,
        real interest rate, and GDP growth.
    """
    return MacroEconomicFetcher().get_macro_context(country_code)    




c:\Users\Personal\AppData\Local\Programs\Python\Python311\Lib\site-packages\langgraph\checkpoint\base\__init__.py:18: LangChainPendingDeprecationWarning: The default value of `allowed_objects` will change in a future version. Pass an explicit value (e.g., allowed_objects='messages' or allowed_objects='core') to suppress this warning.
  from langgraph.checkpoint.serde.jsonplus import JsonPlusSerializer


In [9]:
result = macro_economic_context_tool.invoke({
    "country_code": "GRC"
})

result

REQUEST URL: https://api.worldbank.org/v2/country/GRC/indicator/FP.CPI.TOTL.ZG?format=json&per_page=100
REQUEST URL: https://api.worldbank.org/v2/country/GRC/indicator/SL.UEM.TOTL.ZS?format=json&per_page=100
REQUEST URL: https://api.worldbank.org/v2/country/GRC/indicator/FR.INR.RINR?format=json&per_page=100
REQUEST URL: https://api.worldbank.org/v2/country/GRC/indicator/NY.GDP.MKTP.KD.ZG?format=json&per_page=100


{'inflation_rate': 2.48,
 'unemployment_rate': 8.54,
 'real_interest_rate': 0.0,
 'gdp_growth': 2.07}

In [10]:
behavioral_scoring_tool.invoke({
    "debtrules3": 2,
    "debtnorms8": 3,
    "debtrules16": 4,
    "debtpers5": 1,
    "debtnorms9": 3,
    "debtpers9": 5,
    "debtpers11": 1,
    "debtrules9": 3,
})


(1.0582999999999998, 0.5916760000000001, 1.216518, 0.031063000000000004)

In [11]:
financial_wellbeing_tool = tool(calculate_financial_wellbeing)

financial_wellbeing_tool.invoke({
    "dti": 0.34,
    "cash_buffer_months": 1.6,
    "savings_rate": 0.07,
    "late_frequency": 0.10,
    "high_interest_debt_share": 0.45,
    "source": "mixed"
})


{'financial_wellbeing_score': 52,
 'category': 'Stressed',
 'breakdown': {'dti_score': 12,
  'cash_buffer_score': 10,
  'savings_rate_score': 10,
  'late_frequency_score': 12,
  'high_interest_debt_score': 8},
 'inputs': {'dti': 0.34,
  'cash_buffer_months': 1.6,
  'savings_rate': 0.07,
  'late_frequency': 0.1,
  'high_interest_debt_share': 0.45,
  'source': 'mixed'}}

c:\Users\Personal\AppData\Local\Programs\Python\Python311\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


ImportError: cannot import name 'ToolExecutor' from 'langgraph.prebuilt' (c:\Users\Personal\AppData\Local\Programs\Python\Python311\Lib\site-packages\langgraph\prebuilt\__init__.py)